# Phase I Shared Baseline: Minimal Verification Contract
**Team SHAP_SHIFTERS | AI and Visualization Lab WS26**

This notebook verifies the precomputed Phase I data artifacts in `shared/data/`.
Both pairs (**Pair A: AI-aided** and **Pair B: Human-only**) start from these exact static tables.

> **Protocol Note for Pair B (Human-Only):**
> This notebook provides only baseline integrity verification and summary cohort sanity checks.
> It **strictly omits** any dashboard code or final analytical plots, ensuring that Pair B has a clean slate to build their independent visual analytics pipeline without generative AI assistance.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Path resolution (works whether executed from repo root or shared/)
DATA_DIR = Path("data") if Path("data").exists() else Path("../shared/data")
print(f"Loading data from: {DATA_DIR.resolve()}")


Loading data from: C:\Users\juba9\PycharmProjects\VIS4AI_SHAP_SHIFTERS\shared\data


## 1. Load Precomputed Data Tables
We verify that all required static tables (`cohort_encounters`, `retrieval_results`, `counterfactual_flips`) and `baseline_metrics.json` are present and readable.


In [ ]:
cohort_df = pd.read_parquet(DATA_DIR / "cohort_encounters.parquet")
retrieval_df = pd.read_parquet(DATA_DIR / "retrieval_results.parquet")
flips_df = pd.read_parquet(DATA_DIR / "counterfactual_flips.parquet")

with open(DATA_DIR / "baseline_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print(f"✓ cohort_encounters.parquet   : {cohort_df.shape[0]:,} rows x {cohort_df.shape[1]} columns")
print(f"✓ retrieval_results.parquet   : {retrieval_df.shape[0]:,} rows x {retrieval_df.shape[1]} columns")
print(f"✓ counterfactual_flips.parquet: {flips_df.shape[0]:,} rows x {flips_df.shape[1]} columns")
print(f"✓ baseline_metrics.json       : Generated {metrics['generated_at']} (Script v{metrics['script_version']})")


✓ cohort_encounters.parquet   : 30,248 rows x 62 columns
✓ retrieval_results.parquet   : 30,248 rows x 21 columns
✓ counterfactual_flips.parquet: 30,248 rows x 17 columns
✓ baseline_metrics.json       : Generated 2026-10-28T22:26:12 (Script v2.0.0)


## 2. Cohort Demographics & Ground Truth Class Balance
Verify cohort definition:
- Filtered strictly to **repeat-admission patients** (`patient_nbr` $\ge 2$).
- Ground truth target label: **Next-admission primary ICD-9 diagnosis** (`diag_1` at visit $V_{t+1}$) mapped to 4 pathways:
  1. `Kidney / Renal` (ICD-9: 580–589, 250.4x)
  2. `Cardiovascular` (ICD-9: 390–459, 250.7x)
  3. `Eye / Neurological` (ICD-9: 250.5x, 250.6x, 357, 362)
  4. `Uncomplicated / Other`
- Final visit of each patient dropped (no subsequent admission to evaluate).
- **Zero label leakage**: Model representations use only observable index-admission clinical fields.


In [ ]:
total_encounters = len(cohort_df)
unique_patients = cohort_df["patient_nbr"].nunique()
pathway_counts = cohort_df["true_next_pathway"].value_counts()
pathway_pcts = cohort_df["true_next_pathway"].value_counts(normalize=True) * 100

cohort_summary = pd.DataFrame({
    "Encounters": pathway_counts,
    "Share (%)": pathway_pcts.map("{:.2f}%".format)
})

print(f"Total Cohort Encounters: {total_encounters:,}")
print(f"Unique Repeat Patients : {unique_patients:,}")
print(f"Mean Visits per Patient: {total_encounters / unique_patients:.2f}")
print("\nComplication Pathway Class Distribution:")
cohort_summary


Total Cohort Encounters: 30,248
Unique Repeat Patients : 16,773
Mean Visits per Patient: 1.80

Complication Pathway Class Distribution:


,Encounters,Share (%)
true_next_pathway,,
Uncomplicated / Other,20073,66.36%
Cardiovascular,8855,29.27%
Kidney / Renal,757,2.50%
Eye / Neurological,563,1.86%


## 3. Baseline Retrieval Performance Comparison
Comparison of:
1. **Model A (Transparent Clinical-Feature KNN)**: StandardScaler on numeric features + One-Hot categorical + Cosine KNN ($k=10$).
2. **Model B (Bio_ClinicalBERT KNN)**: 768-D mean-pooled embeddings over serialized authentic encounter fields + Cosine KNN ($k=10$).
3. **Majority Class Baseline**: Always predicts the modal pathway (*Uncomplicated / Other*).
4. **Random Neighbor Baseline**: Uniform random sampling of 10 other patients.

*Rule: Neighbors are strictly drawn from OTHER patients (no self-matches or intra-patient leakage).*


In [ ]:
summary_table = pd.DataFrame([
    {
        "System / Model": "Model A (Clinical Count / Features KNN)",
        "Input Representation": "Standardized numeric + One-hot categorical (127-D)",
        "Accuracy": f"{metrics['models']['model_a']['accuracy']:.1%}",
        "Macro Recall": f"{metrics['models']['model_a']['macro_recall']:.1%}",
        "Mean Agreement (k=10)": f"{metrics['models']['model_a']['mean_agreement']:.3f}",
        "Hit@10": f"{metrics['models']['model_a'].get('hit_at_k', 0):.1%}",
    },
    {
        "System / Model": "Model B (Bio_ClinicalBERT KNN)",
        "Input Representation": "768-D mean-pooled clinical embeddings",
        "Accuracy": f"{metrics['models']['model_b']['accuracy']:.1%}",
        "Macro Recall": f"{metrics['models']['model_b']['macro_recall']:.1%}",
        "Mean Agreement (k=10)": f"{metrics['models']['model_b']['mean_agreement']:.3f}",
        "Hit@10": f"{metrics['models']['model_b'].get('hit_at_k', 0):.1%}",
    },
    {
        "System / Model": "Majority Class Baseline",
        "Input Representation": f"Constant prediction ('{metrics['models']['majority_baseline']['majority_class']}')",
        "Accuracy": f"{metrics['models']['majority_baseline']['accuracy']:.1%}",
        "Macro Recall": f"{metrics['models']['majority_baseline']['macro_recall']:.1%}",
        "Mean Agreement (k=10)": "N/A",
        "Hit@10": "N/A",
    },
    {
        "System / Model": "Random Neighbor Baseline",
        "Input Representation": "Uniform random choice over other patients",
        "Accuracy": f"{metrics['models']['random_neighbor_baseline']['accuracy']:.1%}",
        "Macro Recall": f"{metrics['models']['random_neighbor_baseline']['macro_recall']:.1%}",
        "Mean Agreement (k=10)": f"{metrics['models']['random_neighbor_baseline']['mean_agreement']:.3f}",
        "Hit@10": f"{metrics['models']['random_neighbor_baseline'].get('hit_at_k', 0):.1%}",
    },
]).set_index("System / Model")

summary_table


,Input Representation,Accuracy,Macro Recall,Mean Agreement (k=10),Hit@10
System / Model,,,,,
Model A (Clinical Count / Features KNN),Standardized numeric + One-hot categorical (127-D),63.0%,26.3%,0.541,96.0%
Model B (Bio_ClinicalBERT KNN),768-D mean-pooled clinical embeddings,63.0%,26.5%,0.539,95.8%
Majority Class Baseline,Constant prediction ('Uncomplicated / Other'),66.4%,25.0%,N/A,N/A
Random Neighbor Baseline,Uniform random choice over other patients,62.1%,24.9%,0.526,95.6%


## 4. Counterfactual Perturbations & Sensitivity Analysis
Evaluates the stability and potential demographic shortcuts of Model A vs. Model B by computing the number of neighbor flips ($0$ to $10$) when the query encounter undergoes:
1. **Sex swap** ($M \leftrightarrow F$)
2. **Age shift** ($+20$ years)
3. **HbA1c elevation** (None / Norm $\to$ $>8$)


In [ ]:
cf_data = []
for edit, label in [
    ("sex_swap", "Sex Swap (Male <-> Female)"),
    ("age_shift", "Age Shift (+20 Years)"),
    ("hba1c_elevation", "HbA1c Elevation (None/Norm -> >8)"),
]:
    n_app = metrics["counterfactual_mean_flips"]["n_applicable"][edit]
    flips_a = metrics["counterfactual_mean_flips"]["model_a"][edit]
    flips_b = metrics["counterfactual_mean_flips"]["model_b"][edit]
    cf_data.append({
        "Perturbation Condition": label,
        "Applicable Encounters": f"{n_app:,}",
        "Model A Mean Flips (/10)": f"{flips_a:.2f}",
        "Model B Mean Flips (/10)": f"{flips_b:.2f}",
        "Disparity (Model B - Model A)": f"{flips_b - flips_a:+.2f}",
    })

cf_table = pd.DataFrame(cf_data).set_index("Perturbation Condition")
cf_table


,Applicable Encounters,Model A Mean Flips (/10),Model B Mean Flips (/10),Disparity (Model B - Model A)
Perturbation Condition,,,,
Sex Swap (Male <-> Female),"30,248",8.86,4.15,-4.71
Age Shift (+20 Years),"29,495",5.42,5.11,-0.31
HbA1c Elevation (None/Norm -> >8),"26,866",4.36,8.22,+3.85


## 5. Verification Contract Check
Verify that `cohort_encounters.parquet` and `retrieval_results.parquet` satisfy all field contracts expected downstream by Pair A and Pair B.


In [ ]:
# Column schema validation
req_cohort = ["encounter_id", "patient_nbr", "gender", "age", "true_next_pathway", "umap_x", "umap_y"]
req_retrieval = ["encounter_id", "model_a_neighbors", "model_b_neighbors", "model_a_agreement", "model_b_agreement", "model_a_pred_pathway", "model_b_pred_pathway"]

assert all(c in cohort_df.columns for c in req_cohort), "Missing columns in cohort_encounters.parquet"
assert all(c in retrieval_df.columns for c in req_retrieval), "Missing columns in retrieval_results.parquet"
assert len(cohort_df) == len(retrieval_df) == len(flips_df), "Mismatched row counts across artifacts"
assert not cohort_df["umap_x"].isna().any(), "Found NaN in UMAP coordinates"

print("=" * 70)
print("VERIFICATION COMPLETE: Shared baseline data contract is 100% valid.")
print("Pair B may now proceed with their human-only visualization workflow.")
print("=" * 70)


VERIFICATION COMPLETE: Shared baseline data contract is 100% valid.
Pair B may now proceed with their human-only visualization workflow.
